In [0]:
%run "/Workspace/utilities"

In [0]:
dbutils.widgets.text("load_date", "")
dbutils.widgets.text("min_support", "0.01")
dbutils.widgets.text("min_confidence", "0.3")

load_date = dbutils.widgets.get("load_date")
min_support = float(dbutils.widgets.get("min_support"))
min_confidence = float(dbutils.widgets.get("min_confidence"))

# COMMAND ----------

# Use config from utilities
logger = DataLogger("market_basket_analysis")

order_items_path = config.get_silver_path("order_items")
products_path = config.get_silver_path("products")
market_basket_path = config.get_gold_path("market_basket")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Read Data

# COMMAND ----------

df_order_items = spark.read.format("delta").load(order_items_path)
df_products = spark.read.format("delta").load(products_path)

# Join to get product names
df_transactions = df_order_items.join(
    df_products.select("product_id", "product_name", "category"),
    "product_id"
)

logger.info(f"Loaded {df_transactions.count()} transaction items")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Prepare Transaction Data

# COMMAND ----------

from pyspark.sql.functions import *

# Create transaction baskets (list of products per order)
df_baskets = df_transactions.groupBy("order_id").agg(
    collect_list("product_name").alias("products"),
    collect_list("category").alias("categories"),
    count("*").alias("basket_size")
)

# Filter baskets with at least 2 items
df_baskets = df_baskets.filter(col("basket_size") >= 2)

logger.info(f"Created {df_baskets.count()} baskets")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Calculate Product Co-occurrence

# COMMAND ----------

from pyspark.sql.functions import explode, col
from itertools import combinations

# Create product pairs from each basket
def create_product_pairs(products_list):
    """Generate all pairs from a list of products"""
    if len(products_list) < 2:
        return []
    return list(combinations(sorted(products_list), 2))

# Register UDF
from pyspark.sql.types import ArrayType, StructType, StructField, StringType
pair_schema = ArrayType(StructType([
    StructField("product_a", StringType()),
    StructField("product_b", StringType())
]))

create_pairs_udf = udf(lambda x: [{"product_a": p[0], "product_b": p[1]} 
                                   for p in create_product_pairs(x)], pair_schema)

# Generate pairs
df_pairs = df_baskets.select(
    "order_id",
    explode(create_pairs_udf("products")).alias("pair")
).select(
    "order_id",
    col("pair.product_a").alias("product_a"),
    col("pair.product_b").alias("product_b")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Calculate Association Metrics

# COMMAND ----------

# Total number of transactions
total_transactions = df_baskets.count()

# Count individual product occurrences
df_product_count = df_transactions.groupBy("product_name").agg(
    count("order_id").alias("product_count")
)

# Count pair occurrences
df_pair_count = df_pairs.groupBy("product_a", "product_b").agg(
    count("*").alias("pair_count")
)

# Calculate support
df_pair_support = df_pair_count.withColumn(
    "support",
    col("pair_count") / total_transactions
)

# Filter by minimum support
df_frequent_pairs = df_pair_support.filter(col("support") >= min_support)

# COMMAND ----------

# Calculate confidence: P(B|A) = P(A and B) / P(A)
df_with_confidence = df_frequent_pairs \
    .join(
        df_product_count.withColumnRenamed("product_name", "product_a")
                       .withColumnRenamed("product_count", "count_a"),
        "product_a"
    ) \
    .withColumn(
        "confidence",
        col("pair_count") / col("count_a")
    ) \
    .filter(col("confidence") >= min_confidence)

# Calculate lift: P(A and B) / (P(A) * P(B))
df_with_lift = df_with_confidence \
    .join(
        df_product_count.withColumnRenamed("product_name", "product_b")
                       .withColumnRenamed("product_count", "count_b"),
        "product_b"
    ) \
    .withColumn(
        "lift",
        (col("pair_count") / total_transactions) / 
        ((col("count_a") / total_transactions) * (col("count_b") / total_transactions))
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## Create Association Rules

# COMMAND ----------

df_rules = df_with_lift.select(
    col("product_a").alias("antecedent"),
    col("product_b").alias("consequent"),
    col("support"),
    col("confidence"),
    col("lift"),
    col("pair_count").alias("transaction_count")
).orderBy("lift", ascending=False)

# Add recommendations
df_rules = df_rules.withColumn(
    "recommendation_strength",
    when(col("lift") >= 3, "Strong")
    .when(col("lift") >= 2, "Moderate")
    .otherwise("Weak")
)

logger.info(f"Generated {df_rules.count()} association rules")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Category-Level Analysis

# COMMAND ----------

# Create category baskets
df_category_baskets = df_transactions.groupBy("order_id").agg(
    collect_set("category").alias("categories")  # Use set to avoid duplicates
)

df_category_baskets = df_category_baskets.filter(size("categories") >= 2)

# Generate category pairs
df_category_pairs = df_category_baskets.select(
    "order_id",
    explode(create_pairs_udf("categories")).alias("pair")
).select(
    "order_id",
    col("pair.product_a").alias("category_a"),
    col("pair.product_b").alias("category_b")
)

# Count category pairs
df_category_rules = df_category_pairs.groupBy("category_a", "category_b").agg(
    count("*").alias("pair_count")
).withColumn(
    "support",
    col("pair_count") / total_transactions
).filter(col("support") >= min_support).orderBy("pair_count", ascending=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Write Results

# COMMAND ----------

# Combine product and category rules
df_final_rules = df_rules.withColumn("rule_type", lit("PRODUCT")) \
    .select(
        "antecedent",
        "consequent",
        "support",
        "confidence",
        "lift",
        "transaction_count",
        "recommendation_strength",
        "rule_type"
    )

df_category_rules_final = df_category_rules \
    .withColumnRenamed("category_a", "antecedent") \
    .withColumnRenamed("category_b", "consequent") \
    .withColumn("confidence", lit(None).cast("double")) \
    .withColumn("lift", lit(None).cast("double")) \
    .withColumn("recommendation_strength", lit("N/A")) \
    .withColumn("rule_type", lit("CATEGORY")) \
    .withColumnRenamed("pair_count", "transaction_count")

# Union both types
df_all_rules = df_final_rules.unionByName(df_category_rules_final, allowMissingColumns=True)

# Add metadata
from datetime import datetime
df_all_rules = df_all_rules.withColumn("analysis_date", lit(datetime.now()))

# Write to Gold layer
try:
    df_all_rules.write \
        .format("delta") \
        .mode("overwrite") \
        .save(market_basket_path)
    
    logger.info(f"✅ Successfully wrote {df_all_rules.count()} market basket rules to Gold layer")
    
except Exception as e:
    logger.error("Failed to write to Gold layer", e)
    raise

# COMMAND ----------

# MAGIC %md
# MAGIC ## Display Top Recommendations

# COMMAND ----------

print("=== Top 20 Product Associations (Highest Lift) ===\n")

top_product_rules = df_rules.filter(col("recommendation_strength") == "Strong") \
    .orderBy("lift", ascending=False) \
    .limit(20)

display(top_product_rules)

# COMMAND ----------

print("\n=== Top Category Associations ===\n")

display(df_category_rules.limit(10))

# COMMAND ----------

# Summary statistics
print("\n=== Market Basket Analysis Summary ===\n")

summary = {
    "Total Baskets Analyzed": df_baskets.count(),
    "Product Rules Generated": df_rules.count(),
    "Strong Recommendations": df_rules.filter(col("recommendation_strength") == "Strong").count(),
    "Category Rules Generated": df_category_rules.count()
}

for key, value in summary.items():
    print(f"{key}: {value:,}")

logger.info("Market basket analysis completed successfully")
